# 05. Runnable 기본 조합 (LCEL)

- LangChain에서의 Runnable은 입력값을 받아 특정 작업을 수행하고 출력값을 반환하는 실행 가능한 구성요소임.
- 이를 활용하면 프롬프트, 모델, 출력 파서, 사용자 정의 함수, 병렬 처리 흐름을 직관적인 파이프라인으로 표현할 수 있음.
- LCEL(LangChain Expression Language)은 `|` 연산자로 컴포넌트를 체인처럼 연결함.
- Runnable 조합을 사람이 읽기 쉬운 파이프라인 문법으로 표현한 것.


```
prompt | llm | parser
```

In [1]:
# 상위 폴더 path 설정
import sys
sys.path.append("..")

from dotenv import load_dotenv
load_dotenv(override=True, dotenv_path="../.env")

from langchain_openai import ChatOpenAI
from common_config import llm_connect
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

temperature = 0
max_tokens = 2086

llm = llm_connect()

## 1. 기본 체인: prompt | llm | parser

**📊 데이터 흐름** — 앞 단계의 출력이 그대로 다음 단계의 입력이 됨 (화살표 위 글자 = 그 사이를 흐르는 데이터 타입)

```mermaid
flowchart LR
    IN["입력 dict<br/>{#quot;topic#quot;:<br/>#quot;벡터 데이터베이스#quot;}"]
    P["prompt<br/>ChatPromptTemplate"]
    L["llm<br/>ChatOpenAI"]
    S["parser<br/>StrOutputParser"]
    OUT["최종 출력<br/>str"]
    IN --> P
    P -->|"ChatPromptValue<br/>(완성된 메시지 목록)"| L
    L -->|"AIMessage<br/>(content + 메타데이터)"| S
    S --> OUT
```

In [2]:
prompt = ChatPromptTemplate.from_messages([
    ("human", "{topic}에 대해 한 문장으로 설명해줘")
])

parser = StrOutputParser()

# | 연산자로 체인 구성
chain = prompt | llm | parser

result = chain.invoke({"topic": "벡터 데이터베이스"})
print(result)

벡터 데이터베이스는 **텍스트나 이미지 같은 데이터를 의미를 반영한 벡터로 저장하고, 비슷한 의미의 데이터를 빠르게 찾는 데이터베이스**입니다.


## 2. 체인 스트리밍
- LangChain 체인 실행 결과를 스트리밍 방식으로 조금씩 받아 출력하는 코드

**📊 invoke() vs stream()** — invoke는 답변이 다 만들어질 때까지 기다리고, stream은 만들어지는 대로 조각(chunk)을 받음

```mermaid
sequenceDiagram
    participant Code as 내 코드
    participant Chain as chain
    Note over Code,Chain: invoke() - 전부 완성된 뒤 한 번에 받음
    Code->>Chain: invoke({"topic": "임베딩"})
    Chain-->>Code: "임베딩은 텍스트를 숫자 벡터로 바꾸는 기술입니다."
    Note over Code,Chain: stream() - 생성되는 즉시 조각 단위로 받음
    Code->>Chain: stream({"topic": "임베딩"})
    Chain-->>Code: "임베딩은"
    Chain-->>Code: " 텍스트를"
    Chain-->>Code: " 숫자 벡터로 ..."
```

In [3]:
for chunk in chain.stream({"topic": "임베딩"}):
    # 현재 받은 응답 조각을 출력 후 줄바꿈 없이, 버퍼에 쌓아두지 않고 즉시 출력
    print(chunk, end="", flush=True)

임베딩은 **텍스트나 이미지 같은 데이터를 컴퓨터가 이해하고 비교할 수 있도록 숫자 벡터로 바꾼 표현**입니다.

In [4]:
for chunk in chain.stream({"topic": "임베딩"}):
    # Python이 출력 내용을 잠시 버퍼에 모아두었다가 나중에 보여줌
    print(chunk, end="", flush=False)

임베딩은 단어나 문장 같은 정보를 컴퓨터가 이해할 수 있도록 비슷한 의미끼리 가까운 숫자 벡터로 바꾸는 방법입니다.

# 3. Runnable의 공통 실행 메서드
- 입력 값을 받아 어떤 작업을 수행하고, 출력값을 반환하는 표준 실행 단위
- 프롬프트, LLM, 출력파서, Retriever, 사용자 정의 함수 등을 하나의 실행 규칙으로 연결하기 위한 표준 인터페이스
- 기본 : 입력 → Runnable → 출력
- 여러 Runnable 연결 : 입력 → prompt → llm → parser → custom function → 최종 출력 

**📊 체인도 하나의 Runnable** — 여러 Runnable을 `|`로 묶으면 그 결과물도 Runnable이라, 바깥에서는 부품 하나처럼 똑같이 호출함

```mermaid
flowchart LR
    IN["입력"] --> chain
    subgraph chain["chain (이것도 Runnable)"]
        direction LR
        P["prompt"] --> L["llm"] --> S["parser"]
    end
    chain --> OUT["출력"]
```

- Runnable의 공통 실행 메서드 

| 메서드                    | 의미              | 사용 상황           |
| ---------------------- | --------------- | --------------- |
| `invoke()`             | 단일 입력 실행        | 가장 기본 실행        |
| `ainvoke()`            | 비동기 단일 실행       | async 환경        |
| `batch()`              | 여러 입력을 한 번에 실행  | 여러 질문 일괄 처리     |
| `abatch()`             | 비동기 batch 실행    | 대량 처리           |
| `stream()`             | 결과를 조각 단위로 스트리밍 | 챗봇 실시간 출력       |
| `astream()`            | 비동기 스트리밍        | 웹소켓, SSE 등      |
| `batch_as_completed()` | 완료되는 순서대로 결과 반환 | 입력별 처리 시간이 다를 때 |


**📊 대표 메서드 3가지의 입출력 형태** — 같은 Runnable이라도 호출하는 메서드에 따라 주고받는 모양이 달라짐 (`a`로 시작하는 메서드는 같은 동작의 비동기 버전)

```mermaid
flowchart LR
    I1["입력 1개"] -->|"invoke()"| O1["출력 1개"]
    I2["입력 리스트<br/>[질문1, 질문2, 질문3]"] -->|"batch()"| O2["출력 리스트<br/>[답1, 답2, 답3]"]
    I3["입력 1개"] -->|"stream()"| O3["출력 조각들<br/>조각1 → 조각2 → 조각3 ..."]
```

## 3. RunnablePassthrough - 입력을 그대로 전달
- RunnablePassthrough는 입력값을 거의 그대로 통과시킴

**📊 RunnablePassthrough** — 받은 값을 손대지 않고 다음 단계로 넘기는 통로 (RAG에서 원래 질문을 검색 결과와 함께 프롬프트까지 넘길 때 주로 쓰임)

```mermaid
flowchart LR
    IN["{#quot;topic#quot;: #quot;RAG#quot;}"] --> PT["RunnablePassthrough<br/>(아무것도 바꾸지 않음)"] --> OUT["{#quot;topic#quot;: #quot;RAG#quot;}"]
```

In [5]:
from langchain_core.runnables import RunnablePassthrough

# 입력값을 그대로 흘려보내기
chain = RunnablePassthrough()
result = chain.invoke({"topic": "RAG"})
print(result)

{'topic': 'RAG'}


- `.assign()`은 기존 딕셔너리에 새로운 필드를 추가할 때 사용함

**📊 RunnablePassthrough.assign()** — 원래 딕셔너리는 그대로 두고, 함수 결과를 새 키로 덧붙임

```mermaid
flowchart LR
    IN["입력<br/>{#quot;word#quot;: #quot;hello#quot;}"]
    F["upper 함수 실행<br/>x[#quot;word#quot;].upper()"]
    OUT["출력<br/>{#quot;word#quot;: #quot;hello#quot;,<br/>#quot;upper#quot;: #quot;HELLO#quot;}"]
    IN -->|"원래 키 그대로 유지"| OUT
    IN --> F
    F -->|"#quot;HELLO#quot;를<br/>upper 키로 추가"| OUT
```

In [6]:
# 실용 예: 입력을 그대로 유지하면서 추가 키 삽입
from langchain_core.runnables import RunnablePassthrough

upper=lambda x: x["word"].upper()
chain = RunnablePassthrough.assign(upper=upper)

result = chain.invoke({"word": "hello"})
print(result)  # {'word': 'hello', 'upper': 'HELLO'}

{'word': 'hello', 'upper': 'HELLO'}


## 4. RunnableLambda - 함수를 Runnable로 감싸기
- RunnableLambda는 일반 파이썬 함수를 Runnable로 감싸는 객체
- RunnableLambda는 체인 중간이나 마지막에 후처리 로직을 넣을 때 주로 사용

**📊 RunnableLambda** — 일반 파이썬 함수를 Runnable로 감싸서 체인의 한 칸(여기서는 마지막 후처리)으로 끼워 넣음

```mermaid
flowchart LR
    IN["{#quot;topic#quot;: #quot;LangGraph#quot;}"] --> P["prompt"] --> L["llm"] --> S["StrOutputParser"]
    S -->|"#quot;LangGraph는 ...#quot;"| RL["RunnableLambda(add_prefix)<br/>일반 파이썬 함수"]
    RL --> OUT["#quot;[결과] LangGraph는 ...#quot;"]
    style RL fill:#fff3cd,stroke:#d4a017,color:#000
```

In [7]:
from langchain_core.runnables import RunnableLambda

# 문자열 -> 문자열 편집 반환 함수
def add_prefix(text: str) -> str:
    return f"[결과] {text}"

prompt = ChatPromptTemplate.from_messages([
    ("human", "{topic}이 뭔가요?")
])

chain = prompt | llm | StrOutputParser() | RunnableLambda(add_prefix)

result = chain.invoke({"topic": "LangGraph"})
print(result)

[결과] LangGraph는 **LLM(대규모 언어 모델) 기반 애플리케이션을 그래프 구조로 설계하고 실행하게 해주는 프레임워크**입니다.  
쉽게 말해, **“에이전트의 흐름을 노드와 엣지로 직접 설계할 수 있는 도구”**라고 보면 됩니다.

## 핵심 개념
- **노드(Node)**: 작업 단위
  - 예: 질문 분석, 검색, 요약, 답변 생성
- **엣지(Edge)**: 노드 간 이동 경로
  - 예: “분석 후 검색으로 이동”
- **상태(State)**: 실행 중 공유되는 데이터
  - 예: 대화 기록, 검색 결과, 중간 추론 결과

## 왜 쓰나요?
일반적인 LLM 앱은 단순히 “입력 → 출력” 흐름인데,  
실제 에이전트는 종종 다음이 필요합니다:

- 조건에 따라 분기
- 여러 단계 반복
- 외부 도구 호출
- 실패 시 재시도
- 사람의 검토(human-in-the-loop)
- 복잡한 메모리/상태 관리

LangGraph는 이런 **복잡한 워크플로우를 안정적으로 구성**하는 데 강합니다.

## LangChain과의 차이
- **LangChain**: 프롬프트, 체인, 도구 연결 등 LLM 앱 개발 전반을 돕는 라이브러리
- **LangGraph**: 그중에서도 특히 **상태ful한 에이전트 흐름, 분기, 루프, 제어 구조**를 그래프로 다루는 데 강함

즉,  
- LangChain은 “구성 요소”에 가깝고
- LangGraph는 “흐름 제어”에 더 초점이 있습니다.

## 간단한 예시
예를 들어 이런 흐름을 만들 수 있습니다:

1. 사용자 질문 수신
2. 질문이 검색이 필요한지 판단
3. 필요하면 검색 도구 호출
4. 검색 결과를 바탕으로 답변 생성
5. 필요 시 다시 검색하거나 사람에게 넘김

이런 구조를 코드로 명확하게 표현할 수 있습니다.

## 한 줄 요약
**LangGraph는 LLM 앱, 특히 에이전트를 “그래프 기반 상태 머신”처럼 만들 수 있게 해주는 프레임워크입니다.**

원하시면 제가 이어서:
1. **LangGraph의

## 5. 병렬 실행 - RunnableParallel

딕셔너리 형태로 여러 체인을 동시에 실행합니다.

**📊 RunnableParallel** — 같은 입력을 여러 체인에 동시에 넣고, 결과를 지정한 키 이름의 딕셔너리로 모아 줌 (걸리는 시간 ≈ 두 체인의 합이 아니라 더 느린 쪽 하나)

```mermaid
flowchart LR
    IN["{#quot;topic#quot;: #quot;RAG#quot;}"]
    subgraph PAR["RunnableParallel"]
        direction TB
        K["korean<br/>prompt_ko<br/>| llm | parser"]
        E["english<br/>prompt_en<br/>| llm | parser"]
    end
    OUT["{#quot;korean#quot;: #quot;RAG는 ...#quot;,<br/>#quot;english#quot;: #quot;RAG is ...#quot;}"]
    IN -->|"같은 입력"| K
    IN -->|"같은 입력"| E
    K --> OUT
    E --> OUT
```

In [8]:
from langchain_core.runnables import RunnableParallel

prompt_ko = ChatPromptTemplate.from_messages([("human", "{topic}을 한국어로 설명해줘")])
prompt_en = ChatPromptTemplate.from_messages([("human", "Explain {topic} in English")])

chain_ko = prompt_ko | llm | StrOutputParser()
chain_en = prompt_en | llm | StrOutputParser()

# 두 체인을 병렬로 실행하여 각각 한국어와 영어 설명 생성
parallel_chain = RunnableParallel(
    korean=chain_ko,
    english=chain_en
)

result = parallel_chain.invoke({"topic": "RAG"})
print("[한국어]", result["korean"])
print("[English]", result["english"])

[한국어] RAG는 **Retrieval-Augmented Generation**의 약자로, 한국어로는 보통 **검색 증강 생성**이라고 합니다.

쉽게 말하면:

- **검색(Retrieval)**: 필요한 정보를 외부 문서나 데이터베이스에서 찾아오고
- **증강(Augmented)**: 그 정보를 모델의 입력에 함께 넣어서
- **생성(Generation)**: 더 정확한 답변을 생성하는 방식입니다.

## 왜 쓰나?
일반 LLM은 학습된 지식만으로 답하기 때문에:
- 최신 정보에 약하고
- 특정 회사 문서, 매뉴얼, 사내 규정 같은 **내부 지식**을 잘 모를 수 있습니다.

RAG는 이런 문제를 줄이기 위해, 질문이 들어오면 먼저 관련 자료를 찾아오고 그 내용을 바탕으로 답하게 합니다.

## 예시
예를 들어 사용자가 이렇게 묻는다면:

> “회사 연차 규정이 뭐야?”

RAG 시스템은 먼저 사내 규정 문서에서 연차 관련 내용을 검색한 뒤,  
그 내용을 참고해서:

> “입사 1년 미만자는 월 개근 시 1일의 연차가 발생하고, 1년 이상자는 …”

처럼 답을 만들어 줍니다.

## 장점
- **최신 정보 반영 가능**
- **도메인 특화 지식 활용 가능**
- **환각(hallucination) 감소**
- 답변의 **근거 제시**가 쉬움

## 한계
- 검색이 잘못되면 답도 잘못될 수 있음
- 문서 품질이 낮으면 성능이 떨어짐
- 검색과 생성 파이프라인이 복잡해질 수 있음

원하시면 제가 이어서 **“RAG 아키텍처를 그림처럼 단계별로 설명”**하거나,  
**“한국어 서비스에 RAG를 적용하는 예시”**도 설명해드릴게요.
[English] RAG stands for **Retrieval-Augmented Generation**.

In simple English, it means:

1. **Retrieve** relevant information from a source  
   - like documents, a database, or the 

## 실습 문제
Q2. RunnableLambda를 활용해 LLM 응답을 받아 단어 수를 세는 함수를 파이프라인 끝에 추가해보세요. 결과로 {'text': '...', 'word_count': 42} 형태로 반환하세요.

In [9]:
from langchain_core.runnables import RunnableLambda

# 1. 프롬프트 구성
prompt = ChatPromptTemplate.from_template(
    "{topic}에 대해 초보자도 이해할 수 있게 설명해줘."
)

# 2. LLM 생성
# llm = ChatOpenAI(
#     model="gpt-4o-mini",
#     temperature=0
# )

# 3. 출력 파서 생성
parser = StrOutputParser()

# 4. LLM 응답을 받아 단어 수를 세는 함수
def count_words(text: str) -> dict:
    return {
        "text": text,
        "word_count": len(text.split())
    }

# 5. RunnableLambda로 함수 감싸기
word_count_runnable = RunnableLambda(count_words)

# 6. LCEL 체인 구성
chain = prompt | llm | parser | word_count_runnable

# 7. 실행
result = chain.invoke({"topic": "LangChain Runnable"})

print(result["word_count"])  # 설명의 단어 수
print("-"*20)
print(result["text"])        # LLM이 생성한 설명

585
--------------------
물론이죠.  
**LangChain의 `Runnable`**을 아주 쉽게 말하면:

> **“데이터를 받아서 처리하고, 다음 단계로 넘길 수 있는 작은 작업 단위”**입니다.

즉, LangChain에서 어떤 일을 할 때  
“입력 → 처리 → 출력” 흐름을 만드는 **기본 블록**이라고 생각하면 됩니다.

---

## 1) 왜 필요한가?

AI 앱을 만들다 보면 보통 이런 작업을 합니다:

1. 사용자의 질문을 받는다
2. 질문을 정리한다
3. LLM에 보낸다
4. 답변을 후처리한다
5. 필요하면 다음 단계로 넘긴다

이런 흐름을 일관되게 만들기 위해 LangChain은 여러 컴포넌트를 **하나의 표준 방식**으로 연결할 수 있게 했고, 그 중심이 바로 `Runnable`입니다.

---

## 2) Runnable을 한 문장으로 이해하기

`Runnable`은:

- **입력(input)** 을 받고
- **출력(output)** 을 내놓는
- **체인처럼 연결 가능한 객체**

입니다.

즉, 함수와 비슷하지만,  
LangChain에서는 이걸 더 체계적으로 만들어서:

- 연결하기 쉽고
- 병렬 실행도 가능하고
- 스트리밍도 지원하고
- 중간값 추적도 편하게

할 수 있게 한 것입니다.

---

## 3) 아주 쉬운 비유

### 비유: 공장 조립 라인
- 첫 번째 기계: 재료를 다듬음
- 두 번째 기계: 조립함
- 세 번째 기계: 포장함

각 기계가 하나의 `Runnable`이라고 보면 됩니다.

각 기계는:
- 들어오는 재료를 받고
- 자기 역할을 수행하고
- 결과를 다음 기계로 넘깁니다.

즉, `Runnable`은 **공장 라인의 한 공정**입니다.

---

## 4) 일반 함수와 뭐가 다를까?

일반 함수도 입력을 받아 출력을 내죠.

예:
```python
def add_one(x):
    return x + 1
```

이것도 사실 “작업 단위”입니다.

하지만 `Runnable`은 여기에